<b><font size="6"> Cars 4 You: Predicting Car Values with ML </font></b><br><br>

`Group 40`

Ana Macedo (20250405)<br>
Catarina Mendinhas (20250422)<br>
Lourenço Silva (20250453)<br>
Maria Fonseca (20250380)<br>

### <font color= '#0400ffff'>**Methodology** </font><a class="anchor" id='top'></a>

- [1. Abstract](#1)
- [2. Import Libraries](#2)
- [3. Metadata](#3)
- [4. Import Datasets](#4)
- [5. Data Cleaning](#5)
    - [5.1. Numeric Features Correction](#5_1)
    - [5.2. Categorical Features Correction](#5_2)
        - [5.2.1. Correct the values](#5_2_1)
    - [5.3. Separate Brands into Cheap and Expensive for Model Comparison](#5_3)
- [6. Data Partition](#6)
- [7. Feature Engineering & Preprocessing](#7)
    - [7.1. Preprocessing on Original Data ](#7_1)
        - [7.1.1. Make a Copy, Treat Outliers and Feature Engineering](#7_1_1)
        - [7.1.2. Divide Dataset into Expensive and Cheap Brands](#7_1_2)
        - [7.1.3. Data Preprocessing of Original Data and Brand Segmented Data](#7_1_3)
    - [7.2. Data Preprocessing Without Outlier Treatment](#7_2)
    - [7.3. Data Preprocessing With Different Scaling Methods](#7_3)
        - [7.3.1. MinMax Scaling](#7_3_1)
        - [7.3.2. MinMax Scaling between -1 and 1](#7_3_2)
        - [7.3.3. Robust Scaling](#7_3_3)
        - [7.3.4. No Scaling](#7_3_4)
    - [7.4. Data Preprocessing With Different Imputation of Missing Values](#7_4)
- [8. Feature Selection](#8)
- [9. Model and Evaluation](#9)
    - [9.1. Model Analysis with different Preprocessing Methods](#9_1)
    - [9.2. Model Analysis for Cheap and Expensive Cars](#9_2)

<a class="anchor" id="1">

# **1. Abstract**

[Back to TOP](#TOP)
</a>

This section presents a structured comparison of regression model performance under different data preprocessing strategies and data segmentations. A fixed subset of selected features, previously identified in the main notebook, is reused throughout this analysis aswell some basic data types corrections, to ensure consistency and avoid redundant procedures.

Multiple preprocessing methods are evaluated, varying key components such as outlier treatment, feature scaling methods, and missing values imputation strategies. Four regression models (RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor and KNeighborsRegressor) already tunned with the original dataset, are trained and evaluated on each preprocessing variant in order to assess how these data preparation choices influence predictive performance and generalization.

In a second stage, the same four models are re-evaluated on brand-based subsets of the data. Brands are grouped into cheap and expensive categories based on their average car price, and these subsets are preprocessed using the same preprocessing techniques as applied to the original dataset. This allows for a direct comparison between global model performance and brand-specific behavior.

Model performance is assessed using R² and Mean Absolute Error (MAE) on training and validation sets, along with respective GAPs. This two-staged experimental design enables a clear interpretation of how preprocessing decisions and data segmentation affect model robustness, stability, and predicitive accuracy across different market segments.

<a class="anchor" id="2">

# **2. Import Libraries**

[Back to TOP](#TOP)
</a>

To embark on this project, it is essential to import the necessary libraries which play a pivotal role in efficient data management and predictive model development. 

This approach employed the use of NumPy and Pandas for the manipulation of data, and Plotly for the creation of interactive visualisations. Statistical tests were conducted using SciPy, while the preprocessing and modelling were dependent on scikit-learn, including tools for imputation, scaling, and a variety of machine learning algorithms. Performance evaluation was carried out using MAE and $R^2$ metrics.

In [1]:
# General Useful Libraries
import pandas as pd
import numpy as np
import os
from math import ceil
import math

# Text Similarity - Categorical Features Correction
from difflib import SequenceMatcher

# Scaling and Encoding
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, TargetEncoder, MinMaxScaler, StandardScaler, RobustScaler

# Missing Value Imputation
from sklearn.impute import KNNImputer, SimpleImputer

# Data Partition
from sklearn.model_selection import train_test_split

#filter methods
from sklearn.feature_selection import VarianceThreshold
from scipy.stats import spearmanr

# spearman 
from sklearn.feature_selection import SelectKBest, f_regression

# mutual information
from sklearn.feature_selection import mutual_info_classif

#wrapper methods
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVC
from sklearn.feature_selection import RFE

# embedded methods
from sklearn.linear_model import Lasso

from sklearn.model_selection import RandomizedSearchCV, PredefinedSplit

# Linear Models
from sklearn.linear_model import Ridge, Lasso, ElasticNet

#KNN
from sklearn.neighbors import KNeighborsRegressor

# Random Forest
from sklearn.ensemble import RandomForestRegressor

#Neural Network
from sklearn.neural_network import MLPRegressor

# DecisionTree
from sklearn.tree import DecisionTreeRegressor

# Ensemble
from sklearn.ensemble import BaggingRegressor, ExtraTreesRegressor, RandomForestRegressor, GradientBoostingRegressor

#Model evaluation
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, median_absolute_error, mean_absolute_percentage_error
import statsmodels.api as sm

# Load Created Functions
import sys
sys.path.append("../")

from Source.visualizations import *
from Source.data_correction_preprocessing import *
from Source.feature_engineering import *
from Source.feature_selection import *
from Source.model_and_assessment import *

# Set random seed for reproducibility
np.random.seed(40111)

<a class="anchor" id="3">

# **3. Metadata**

[Back to TOP](#TOP)
</a>

Understanding the features is essential to interpret the data correctly and conduct the subsequent analysis and modeling steps. This section provides a brief description of each feature in the dataset, explaining its meaning and type.

**carID:** An attribute that contains an identifier for each car.

**Brand:** The car's main brand (e.g., Ford, Toyota).

**model:** The car model.

**year:** The year of registration of the car.

**price:** The car's price when purchased by Cars 4 You (in £).

**transmission:** The car's type of transmission.

**mileage:** The total reported distance travelled by the car (in miles).

**tax:** The amount of road tax (in £) that, in 2020, was applicable to the car in question.

**fuelType:** The type of fuel used by the car (Diesel, Petrol, Hybrid, Electric).

**mpg:** The car's consumption of fuel expressed in average miles per gallon.

**engineSize:** Size of the engine in liters (Cubic Decimeters).

**paintQuality%** The mechanic’s assessment of the cars’ overall paint quality and hull integrity (filled by the mechanic during evaluation). 

**previousOwner:** Number of previous registered owners of the vehicle.

**hasDamage:** Boolean marker filled by the seller at the time of registration stating whether the car is damaged or not.

<a class="anchor" id="4">

# **4. Import Datasets**

[Back to TOP](#TOP)
</a>

Here, we are importing with pd.read_csv() the files to start our project and converting them into Pandas DataFrames. CarID was set to index since it's the unique identifier in all data sets for each individual car.

In [2]:
train = pd.read_csv('../../data/train.csv')
test = pd.read_csv('../../data/test.csv')

train.set_index('carID', inplace = True)
test.set_index('carID', inplace = True)

<a class="anchor" id="5">

# **5. Data Cleaning**

[Back to TOP](#TOP)
</a>

In this section, the objective is to clean the dataset by correcting inconsistencies identified during the exploratory data analysis (EDA) phase, across both numerical and categorical features. This process is supported by a dedicated .py file, which contains a set of helper functions developed to streamline and standardize the data cleaning workflow. Since the analysis where deeply explained in other notebooks, this one will only replicate the corrections.

<a class="anchor" id="5_1">

## **5.1** Numeric Features Correction
[Back to TOP](#TOP)
</a>

**Correct incorrect negative values, round floats that should be integers and turn HasDamage into a boolean**

In [3]:
train = correct_metric_features(train)
test = correct_metric_features(test)

<a class="anchor" id="5_2">

## **5.2** Categorical Features Correction

[Back to TOP](#TOP)

<a class="anchor" id="5_2_1">

### **5.2.1** Correct the values

[Back to TOP](#TOP)

**Correct values with misspelled words for categorical**

In [4]:
# Correct 'Brand' column
train = clean_with_diff(train, 'Brand', threshold_short=0.6, threshold_long=0.6)
test = clean_with_diff(test, 'Brand', threshold_short=0.6, threshold_long=0.6)

# Correct 'transmission' column
train = clean_with_diff(train, 'transmission', threshold_short=0.7, threshold_long=0.7)
test = clean_with_diff(test, 'transmission', threshold_short=0.7, threshold_long=0.7)

# Correct 'model' column
train = clean_with_diff(train, 'model', threshold_short=0.70, threshold_long=0.90)
test = clean_with_diff(test, 'model', threshold_short=0.70, threshold_long=0.90)

# Correct 'fuelType' column
train = clean_with_diff(train, 'fuelType', threshold_short=0.8, threshold_long=0.8)
test = clean_with_diff(test, 'fuelType', threshold_short=0.8, threshold_long=0.8)

**Replace Other by Unknown in 'transmission'**

In [5]:
train = replace_category_transmission(train)
test = replace_category_transmission(test)

<a class="anchor" id="5_3">

## **5.3** Separate Brands into Cheap and Expensive for Model Comparison

[Back to TOP](#TOP)

One of the analysis conducted in this notebook investigates whether the models achieve different predicitve performance across brands. To support this analysis, brands are first segmented according to their average price in the training data. 

Brands with an average price equal or above 20.000€ are classified as expensive, while brands with an average price below 20.000€ are classified as cheap. This segmentation is performed prior to data partitioning to ensure a consistent and unbiased definition of price-based brand groups.

In [6]:
avg_price_brand = train.groupby('Brand')['price'].mean()

expensive_brands = avg_price_brand[avg_price_brand >= 20_000].index.tolist()
cheap_brands = avg_price_brand[avg_price_brand < 20_000].index.tolist()

<a class="anchor" id="6">

# **6. Data Partition**

[Back to TOP](#TOP)

In this section, the dataset is partitioned into independent and dependent variables, where the independent features are defined as X and the target variable (price) as y. This separation is essential to clearly distinguish the inputs used by the model from the value it is expected to predict.

Subsequently, the data is split into training and validation sets using the holdout method. This step allows the models to be trained on a subset of the data while being evaluated on unseen observations, providing a more realistic assessment of their generalization capability. This is also a critical measure to prevent data leakage, which occurs when information from the validation set influences the training process. 

After the split, all preprocessing steps that rely on statistical properties of the data (such as outlier detection, imputation, or scaling) are computed exclusively using the training set and then applied to the validation set.

This approach ensures that model evaluation remains unbiased and that the reported performance metrics accurately reflect how the model would behave when exposed to new, unseen data.

In [7]:
X = train.drop('price', axis = 1)
y = train['price']

X_train, X_val, y_train, y_val = train_test_split(X,y, test_size = 0.2, random_state = 42,  shuffle = True)

<a class="anchor" id="7">

# **7. Feature Engineering & Preprocessing**

[Back to TOP](#TOP)

Multiple preprocessing methods were explored in order to assess the impact of different data preparation strategies on model performance. Each dataset variant applies a specific combination of preprocessing steps, while keeping others factors fixed.

The original preprocessing includes **Outlier Treatment**, **Feature Engineering**, **Categorical Encoding**, **Standard Scaling** and **KNN Missing Values Imputation**. This configuration serves as reference for further model comparisons.

To evaluate the contribution of individual preprocessing components, several alternative datasets were created by selectively modifying this original configuration.These variations include:

- Removing outlier treatment, while keeping all other preprocessing steps unchanged.

- Disabling feature scaling, to assess the effect of scaling on model performance.

- Applying different scaling techniques, namely MinMax, MinMax between -1 and 1, and Robust Scaling.

- Replacing KNN Imputation with Simple Imputation, in order to compare the affect of different missing value strategies.

In addition, two brand-based subsets were created based on average car price: Cheap Brands and Expensive Brands. These subsets were defined prior to data partitioning and were subsequently processed using the same preprocessing as applied to the original dataset, ensuring consistency and comparability across analyses.

<a class="anchor" id="7_1">

## **7.1** Preprocessing on Original Data

[Back to TOP](#TOP)

<a class="anchor" id="7_1_1">

### **7.1.1** Make a Copy, Treat Outliers and Feature Engineering

[Back to TOP](#TOP)

In [8]:
# Original Data Sets
X_train_original = X_train.copy()
y_train_original = y_train.copy()
X_val_original = X_val.copy()
y_val_original = y_val.copy()

In [9]:
# Outliers Treatment
X_train_clean = treat_outliers_custom(X_train_original, X_train_original)
X_val_clean = treat_outliers_custom(X_train_original, X_val_original)

# Update Original Data Sets with cleaned versions
X_train_original = X_train_clean.copy()
X_val_original = X_val_clean.copy()

In [10]:
# Feature Engineering
X_train_fe = create_features(X_train_original, X_train_original, current_year=2020, threshold=3)
X_val_fe = create_features(X_train_original, X_val_original, current_year=2020, threshold=3)

# Update Original Data Sets to include new features
X_train_original = X_train_fe.copy()  
X_val_original = X_val_fe.copy()

<a class="anchor" id="7_1_2">

### **7.1.2** Divide Data Set into Expensive and Cheap Brands

[Back to TOP](#TOP)

In [11]:
X_train_expensive = X_train_original[X_train_original['Brand'].isin(expensive_brands)].copy()
y_train_expensive = y_train_original[X_train_original['Brand'].isin(expensive_brands)].copy()

X_train_cheap = X_train_original[X_train_original['Brand'].isin(cheap_brands)].copy()
y_train_cheap = y_train_original[X_train_original['Brand'].isin(cheap_brands)].copy()

X_val_expensive = X_val_original[X_val_original['Brand'].isin(expensive_brands)].copy()
y_val_expensive = y_val_original[X_val_original['Brand'].isin(expensive_brands)].copy()

X_val_cheap = X_val_original[X_val_original['Brand'].isin(cheap_brands)].copy()
y_val_cheap = y_val_original[X_val_original['Brand'].isin(cheap_brands)].copy()

<a class="anchor" id="7_1_3">

### **7.1.3** Data Preprocessing of Original Data and Brand Segmented Data

[Back to TOP](#TOP)

In [12]:
# Encoding, Scaling and Missing Value Imputation
X_train_dp_original = data_preprocessing(X_train_original, y_train_original, X_train_original, neighbors=5, imputation_method="knn", scaling_method="standard")
X_val_dp_original = data_preprocessing(X_train_original, y_train_original, X_val_original, neighbors=5, imputation_method="knn", scaling_method="standard")

# Update Original Data Sets to include Preprocessing
X_train_original = X_train_dp_original.copy()
X_val_original = X_val_dp_original.copy()

In [13]:
# Encoding, Scaling and Missing Value Imputation
X_train_dp_expensive = data_preprocessing(X_train_expensive, y_train_expensive, X_train_expensive, neighbors=5, imputation_method="knn", scaling_method="standard")
X_val_dp_expensive = data_preprocessing(X_train_expensive, y_train_expensive, X_val_expensive, neighbors=5, imputation_method="knn", scaling_method="standard")

X_train_dp_cheap = data_preprocessing(X_train_cheap, y_train_cheap, X_train_cheap, neighbors=5, imputation_method="knn", scaling_method="standard")
X_val_dp_cheap = data_preprocessing(X_train_cheap, y_train_cheap, X_val_cheap, neighbors=5, imputation_method="knn", scaling_method="standard")

# Update Data Sets to include Preprocessing
X_train_expensive = X_train_dp_expensive.copy()
X_val_expensive = X_val_dp_expensive.copy()
X_train_cheap = X_train_dp_cheap.copy()
X_val_cheap = X_val_dp_cheap.copy()

<a class="anchor" id="7_2">

## **7.2** Data Preprocessing Without Outlier Treatment

[Back to TOP](#TOP)

In [14]:
# Data Sets to not treat outliers
X_train_with_outliers = X_train.copy()
y_train_with_outliers = y_train.copy()
X_val_with_outliers = X_val.copy()
y_val_with_outliers = y_val.copy()

In [15]:
# Feature Engineering
X_train_fe_wo = create_features(X_train_with_outliers, X_train_with_outliers, current_year=2020, threshold=3)
X_val_fe_wo = create_features(X_train_with_outliers, X_val_with_outliers, current_year=2020, threshold=3)

# Update Data Sets to include new features
X_train_with_outliers = X_train_fe_wo.copy()  
X_val_with_outliers = X_val_fe_wo.copy()

In [16]:
# Encoding, Scaling and Missing Value Imputation
X_train_dp_outliers = data_preprocessing(X_train_with_outliers, y_train_with_outliers, X_train_with_outliers, neighbors=5, imputation_method="knn", scaling_method="standard")
X_val_dp_outliers = data_preprocessing(X_train_with_outliers, y_train_with_outliers, X_val_with_outliers, neighbors=5, imputation_method="knn", scaling_method="standard")

# Update Data Sets to include Preprocessing
X_train_with_outliers = X_train_dp_outliers.copy()
X_val_with_outliers = X_val_dp_outliers.copy()

<a class="anchor" id="7_3">

## **7.3** Data Preprocessing With Different Scaling Methods

[Back to TOP](#TOP)

<a class="anchor" id="7_3_1">

### **7.3.1** MinMax Scaling

[Back to TOP](#TOP)

In [17]:
# Data Sets to use Min-Max Scaling
X_train_minmax = X_train.copy()
y_train_minmax = y_train.copy()
X_val_minmax = X_val.copy()
y_val_minmax = y_val.copy()

In [18]:
# Treat Outliers
X_train_clean_minmax = treat_outliers_custom(X_train_minmax, X_train_minmax)
X_val_clean_minmax = treat_outliers_custom(X_train_minmax, X_val_minmax)

# Update Data Sets with cleaned versions
X_train_minmax = X_train_clean_minmax.copy()
X_val_minmax = X_val_clean_minmax.copy()

In [19]:
# Feature Engineering
X_train_fe_minmax = create_features(X_train_minmax, X_train_minmax, current_year=2020, threshold=3)
X_val_fe_minmax = create_features(X_train_minmax, X_val_minmax, current_year=2020, threshold=3)

# Update dataframes to include new features
X_train_minmax = X_train_fe_minmax.copy()  
X_val_minmax = X_val_fe_minmax.copy()

In [20]:
# Encoding, Scaling and Missing Value Imputation
X_train_dp_minmax = data_preprocessing(X_train_minmax, y_train_minmax, X_train_minmax, neighbors=5, imputation_method="knn", scaling_method="minmax")
X_val_dp_minmax = data_preprocessing(X_train_minmax, y_train_minmax, X_val_minmax, neighbors=5, imputation_method="knn", scaling_method="minmax")

# Update Data Sets to include Preprocessing
X_train_minmax = X_train_dp_minmax.copy()
X_val_minmax = X_val_dp_minmax.copy()

<a class="anchor" id="7_3_2">

### **7.3.2** MinMax Scaling between -1 and 1

[Back to TOP](#TOP)

In [21]:
# Data sets to use Min-Max Scaling between -1 and 1
X_train_minmax2 = X_train.copy()
y_train_minmax2 = y_train.copy()
X_val_minmax2 = X_val.copy()
y_val_minmax2 = y_val.copy()

In [22]:
# Data sets with outlier treatment
X_train_clean_minmax2 = treat_outliers_custom(X_train_minmax2, X_train_minmax2)
X_val_clean_minmax2 = treat_outliers_custom(X_train_minmax2, X_val_minmax2)

# Update original datasets with cleaned versions
X_train_minmax2 = X_train_clean_minmax2.copy()
X_val_minmax2 = X_val_clean_minmax2.copy()

In [23]:
# Feature Engineering
X_train_fe_minmax2 = create_features(X_train_minmax2, X_train_minmax2, current_year=2020, threshold=3)
X_val_fe_minmax2 = create_features(X_train_minmax2, X_val_minmax2, current_year=2020, threshold=3)

# Update dataframes to include new features
X_train_minmax2 = X_train_fe_minmax2.copy()  
X_val_minmax2 = X_val_fe_minmax2.copy()

In [24]:
# Encoding, Scaling and Missing Value Imputation
X_train_dp_minmax2 = data_preprocessing(X_train_minmax2, y_train_minmax2, X_train_minmax2, neighbors=5, imputation_method="knn", scaling_method="minmax2")
X_val_dp_minmax2 = data_preprocessing(X_train_minmax2, y_train_minmax2, X_val_minmax2, neighbors=5, imputation_method="knn", scaling_method="minmax2")

# Update Data Sets to include Preprocessing
X_train_minmax2 = X_train_dp_minmax2.copy()
X_val_minmax2 = X_val_dp_minmax2.copy()

<a class="anchor" id="7_3_3">

### **7.3.3** Robust Scaling

[Back to TOP](#TOP)

In [25]:
# Data sets without outlier treatment
X_train_robust = X_train.copy()
y_train_robust = y_train.copy()
X_val_robust = X_val.copy()
y_val_robust = y_val.copy()

In [26]:
# Data sets with outlier treatment
X_train_clean_robust = treat_outliers_custom(X_train_robust, X_train_robust)
X_val_clean_robust = treat_outliers_custom(X_train_robust, X_val_robust)

# Update original datasets with cleaned versions
X_train_robust = X_train_clean_robust.copy()
X_val_robust = X_val_clean_robust.copy()

In [27]:
# Feature Engineering
X_train_fe_robust = create_features(X_train_robust, X_train_robust, current_year=2020, threshold=3)
X_val_fe_robust = create_features(X_train_robust, X_val_robust, current_year=2020, threshold=3)

# Update dataframes to include new features
X_train_robust = X_train_fe_robust.copy()  
X_val_robust = X_val_fe_robust.copy()

In [28]:
# Encoding, Scaling and Missing Value Imputation
X_train_dp_robust = data_preprocessing(X_train_robust, y_train_robust, X_train_robust, neighbors=5, imputation_method="knn", scaling_method="robust")
X_val_dp_robust = data_preprocessing(X_train_robust, y_train_robust, X_val_robust, neighbors=5, imputation_method="knn", scaling_method="robust")

# Update Data Sets to include Preprocessing
X_train_robust = X_train_dp_robust.copy()
X_val_robust = X_val_dp_robust.copy()

<a class="anchor" id="7_3_4">

### **7.3.4** No Scaling

[Back to TOP](#TOP)

In [29]:
# Data sets without outlier treatment
X_train_no_scaling = X_train.copy()
y_train_no_scaling = y_train.copy()
X_val_no_scaling = X_val.copy()
y_val_no_scaling = y_val.copy()

In [30]:
# Data sets with outlier treatment
X_train_clean_no_scaling = treat_outliers_custom(X_train_no_scaling, X_train_no_scaling)
X_val_clean_no_scaling = treat_outliers_custom(X_train_no_scaling, X_val_no_scaling)

# Update original datasets with cleaned versions
X_train_no_scaling = X_train_clean_no_scaling.copy()
X_val_no_scaling = X_val_clean_no_scaling.copy()

In [31]:
# Feature Engineering
X_train_fe_no_scaling = create_features(X_train_no_scaling, X_train_no_scaling, current_year=2020, threshold=3)
X_val_fe_no_scaling = create_features(X_train_no_scaling, X_val_no_scaling, current_year=2020, threshold=3)

# Update dataframes to include new features
X_train_no_scaling = X_train_fe_no_scaling.copy()  
X_val_no_scaling = X_val_fe_no_scaling.copy()

In [32]:
# Encoding, Scaling and Missing Value Imputation
X_train_dp_no_scaling = data_preprocessing(X_train_no_scaling, y_train_no_scaling, X_train_no_scaling, neighbors=5, imputation_method="knn", scaling_method="none")
X_val_dp_no_scaling = data_preprocessing(X_train_no_scaling, y_train_no_scaling, X_val_no_scaling, neighbors=5, imputation_method="knn", scaling_method="none")

# Update Data Sets to include Preprocessing
X_train_no_scaling = X_train_dp_no_scaling.copy()
X_val_no_scaling = X_val_dp_no_scaling.copy()

<a class="anchor" id="7_4">

## **7.4** Data Preprocessing With Different Imputation of Missing Values

[Back to TOP](#TOP)

In [33]:
# Data sets without outlier treatment
X_train_simple = X_train.copy()
y_train_simple = y_train.copy()
X_val_simple = X_val.copy()
y_val_simple = y_val.copy()

In [34]:
# Data sets with outlier treatment
X_train_clean_simple = treat_outliers_custom(X_train_simple, X_train_simple)
X_val_clean_simple = treat_outliers_custom(X_train_simple, X_val_simple)

# Update original datasets with cleaned versions
X_train_simple = X_train_clean_simple.copy()
X_val_simple = X_val_clean_simple.copy()

In [35]:
# Feature Engineering
X_train_fe_simple = create_features(X_train_simple, X_train_simple, current_year=2020, threshold=3)
X_val_fe_simple = create_features(X_train_simple, X_val_simple, current_year=2020, threshold=3)

# Update dataframes to include new features
X_train_simple = X_train_fe_simple.copy()  
X_val_simple = X_val_fe_simple.copy()

In [36]:
# Encoding, Scaling and Missing Value Imputation
X_train_dp_simple = data_preprocessing(X_train_simple, y_train_simple, X_train_simple, neighbors=5, imputation_method="simple", scaling_method="standard")
X_val_dp_simple = data_preprocessing(X_train_simple, y_train_simple, X_val_simple, neighbors=5, imputation_method="simple", scaling_method="standard")

# Update Data Sets to include Preprocessing
X_train_simple = X_train_dp_simple.copy()
X_val_simple = X_val_dp_simple.copy()

<a class="anchor" id="1">

# **8. Feature Selection**

[Back to TOP](#TOP)
</a>


For the purpose of this analysis, feature selection is not re-computed for each dataset variant. Instead, the list of selected features obtained on the main notebook is directly reused and applied consistently across all preprocessed datasets and data subsets.

This approach ensures comparability between experiments while avoiding redundant feature selection procedures already explored in the primary analysis.

In [37]:
selected_features = ['year', 'mileage', 'mpg', 'engineSize', 'is_recent_car', 'is_hybrid_or_electric', 'is_automatic', 
                     'fuel_efficiency_score', 'tax_to_engine_ratio', 'brand_median_mileage', 'brand_avg_engineSize', 
                     'fueltype_avg_mpg', 'brand_avg_age', 'Brand_target', 'model_target', 'transmission_Semi-Auto_ohe']

In [47]:
# Feature Selection - Keep only selected features for all datasets
X_train_original = X_train_original[selected_features]
X_val_original = X_val_original[selected_features]

X_train_with_outliers = X_train_with_outliers[selected_features]
X_val_with_outliers = X_val_with_outliers[selected_features]

X_train_minmax = X_train_minmax[selected_features]
X_val_minmax = X_val_minmax[selected_features]

X_train_minmax2 = X_train_minmax2[selected_features]
X_val_minmax2 = X_val_minmax2[selected_features]

X_train_robust = X_train_robust[selected_features]
X_val_robust = X_val_robust[selected_features]

X_train_simple = X_train_simple[selected_features]
X_val_simple = X_val_simple[selected_features]

X_train_no_scaling = X_train_no_scaling[selected_features]
X_val_no_scaling = X_val_no_scaling[selected_features]

X_train_expensive = X_train_expensive[selected_features]
X_val_expensive = X_val_expensive[selected_features]

X_train_cheap = X_train_cheap[selected_features]
X_val_cheap = X_val_cheap[selected_features]

<a class="anchor" id="9">

# **9. Model and Evaluation**

[Back to TOP](#TOP)
</a>

<a class="anchor" id="9_1">

## **9.1. Model Analysis with different Preprocessing Methods**

[Back to TOP](#TOP)
</a>

In the modeling phase, several regression models are evaluated across different preprocessing pipelines and brand-based data subsets. Preprocessed datasets are organized into dictionaries and consistently evaluated using the same training and validation splits. Each model is first assessed across preprocessing variants and subsequently across brand subsets (Original, Cheap, and Expensive), with performance always compared against the Original dataset as a baseline.

In [ ]:
# Dictionary with all preprocessed data sets
data_set_preprocessed = {
    'Original': (X_train_original, y_train_original, X_val_original, y_val_original),
    'Without_Outlier_Treatment': (X_train_with_outliers, y_train_with_outliers, X_val_with_outliers, y_val_with_outliers),
    'MinMax': (X_train_minmax, y_train_minmax, X_val_minmax, y_val_minmax),
    'MinMax2': (X_train_minmax2, y_train_minmax2, X_val_minmax2, y_val_minmax2),
    'Robust': (X_train_robust, y_train_robust, X_val_robust, y_val_robust),
    'No Scaling': (X_train_no_scaling, y_train_no_scaling, X_val_no_scaling, y_val_no_scaling),
    'Simple_Imputation': (X_train_simple, y_train_simple, X_val_simple, y_val_simple)
}

In [ ]:
# Analysis for all preprocessed data sets with RandomForestRegressor
model_rf = RandomForestRegressor(n_estimators=100, min_samples_split=8, min_samples_leaf=5, max_samples=0.9, max_features='log2', max_depth=None, ccp_alpha=0.0, random_state=42)

compare_model_dp(data_set_preprocessed, model_rf)

,Train_MAE,Val_MAE,Gap_MAE_%,Train_R2,Val_R2
Data_Preprocessing,,,,,
Robust,1173.164188,1362.697226,16.155713,0.953950,0.945915
Without_Outlier_Treatment,1173.237852,1363.145903,16.186662,0.953553,0.945676
Original,1176.505460,1369.885429,16.436810,0.953623,0.944211
MinMax2,1182.552621,1371.125783,15.946281,0.952791,0.944312
MinMax,1182.406790,1380.104741,16.719961,0.953250,0.943940
No Scaling,1186.836141,1384.852162,16.684361,0.952436,0.942601
Simple_Imputation,1215.167660,1401.658913,15.346957,0.950910,0.941101


In [ ]:
# Analysis for all preprocessed data sets with ExtraTreesRegressor
model_et = ExtraTreesRegressor(n_estimators=200, min_samples_split=8, min_samples_leaf=1, max_samples=0.6, max_features=0.5, max_depth=30, criterion='squared_error', bootstrap=True, random_state=42)

compare_model_dp(data_set_preprocessed, model_et)

,Train_MAE,Val_MAE,Gap_MAE_%,Train_R2,Val_R2
Data_Preprocessing,,,,,
Without_Outlier_Treatment,1088.745518,1333.057312,22.439752,0.961329,0.947328
Robust,1085.129759,1334.358578,22.967651,0.961790,0.947105
Original,1086.620958,1336.502778,22.996227,0.961867,0.946629
MinMax,1094.427111,1342.692034,22.684464,0.961139,0.945775
MinMax2,1095.478774,1344.047746,22.690442,0.961055,0.945873
No Scaling,1096.302841,1353.483427,23.458900,0.960781,0.944172
Simple_Imputation,1122.360386,1369.224227,21.995060,0.959410,0.942058


In [ ]:
# Analysis for all preprocessed data sets with GradientBoostingRegressor
model_gb = GradientBoostingRegressor(subsample=0.95, n_estimators=1000, min_samples_split=15, min_samples_leaf=5, max_features=0.7, max_depth=5, loss='huber', learning_rate=0.06, random_state=42)

compare_model_dp(data_set_preprocessed, model_gb)

,Train_MAE,Val_MAE,Gap_MAE_%,Train_R2,Val_R2
Data_Preprocessing,,,,,
Without_Outlier_Treatment,1184.043654,1315.129547,11.071035,0.957921,0.951193
Robust,1188.891385,1321.562996,11.159271,0.958626,0.950225
Original,1189.348326,1323.665549,11.293346,0.958551,0.950099
MinMax2,1189.021245,1324.985038,11.434934,0.958023,0.950077
MinMax,1195.306578,1329.468509,11.224060,0.957794,0.949906
No Scaling,1191.511080,1331.468813,11.746238,0.957726,0.948894
Simple_Imputation,1207.401368,1334.999582,10.568003,0.957431,0.948287


In [ ]:
# Analysis for all preprocessed data sets with KNeighborsRegressor
model_knn = KNeighborsRegressor(n_neighbors=10, weights='uniform', p=1, metric='minkowski', leaf_size=20, algorithm='kd_tree')

compare_model_dp(data_set_preprocessed, model_knn)

,Train_MAE,Val_MAE,Gap_MAE_%,Train_R2,Val_R2
Data_Preprocessing,,,,,
Robust,1304.631128,1440.711069,10.430530,0.941710,0.935742
Original,1301.857906,1447.494084,11.186795,0.942305,0.935130
Without_Outlier_Treatment,1313.710482,1460.008878,11.136274,0.940621,0.933639
Simple_Imputation,1336.937767,1487.881013,11.290222,0.939396,0.929156
MinMax2,1400.213949,1553.869016,10.973685,0.931895,0.924887
MinMax,1403.630965,1559.402349,11.097745,0.930927,0.923629
No Scaling,2330.778703,2566.545837,10.115381,0.838843,0.804556


**Effects of Data Preprocessing Methods in Models Conclusion:**

Across all evaluated models, preprocessing choices exhibit a moderate but consistent impact on predictive performance. While preprocessing does not drastically change overall models results, certain decisions, particularly feature scaling, play a significant role in refining model accuracy and generalization. Among the tested scaling techniques, Robust Scaling consistently delivers the most stable and reliable results across different models, but it's never too different than our original, Standard Scaler. The decision to exclude outlier treatment shows a smaller but still noticeable effect, especially in tree-based models.

For the Random Forest and Extra Trees regressors, performance differences across preprocessing variants are relatively limited, confirming the robustness of ensemble tree-based models to data transformations. In both cases, robust scaling slightly improves validation performance, while simple imputation consistently underperforms.

The Gradient Boosting model demonstrates the most stable generalization behavior, exhibiting the smallest performance gaps between training and validation across preprocessing configurations. This suggests that Gradient Boosting is less sensitive to preprocessing variations and benefits from its inherent regularization mechanisms.

In contrast, the K-Nearest Neighbors regressor is highly sensitive to preprocessing decisions, specially scaling. Feature scaling proves to be critical for acceptable performance, with unscaled data leading to a substantial degradation in both MAE and R². This behavior highlights the strong dependence of distance-based models on appropriate feature scaling.

Overall, these results reinforce that effective preprocessing should be aligned with model characteristics. While tree-based models are relatively robust to preprocessing choices, distance-based models require careful data preparation. Preprocessing therefore acts as a performance refinement tool rather than a substitute for appropriate model selection.

<a class="anchor" id="9_2">

## **9.2. Model Analysis for Cheap and Expensive Cars**

[Back to TOP](#TOP)
</a>

In [ ]:
# Dictionary with data sets by brand category
data_set_brands = {
    'Original': (X_train_original, y_train_original, X_val_original, y_val_original),
    'Expensive': (X_train_expensive, y_train_expensive, X_val_expensive, y_val_expensive),
    'Cheap': (X_train_cheap, y_train_cheap, X_val_cheap, y_val_cheap)
}

In [ ]:
# Analysis for all brand subsets with RandomForestRegressor
model_rf = RandomForestRegressor(n_estimators=100, min_samples_split=8, min_samples_leaf=5, max_samples=0.9, max_features='log2', max_depth=None, ccp_alpha=0.0, random_state=42)

compare_model_dp(data_set_brands, model_rf)

,Train_MAE,Val_MAE,Gap_MAE_%,Train_R2,Val_R2
Data_Preprocessing,,,,,
Cheap,869.845136,1011.411636,16.274908,0.951140,0.937701
Original,1176.505460,1369.885429,16.436810,0.953623,0.944211
Expensive,1729.982410,1984.276516,14.699231,0.931231,0.920792


In [ ]:
# Analysis for all brand subsets with ExtraTreesRegressor
model_et = ExtraTreesRegressor(n_estimators=200, min_samples_split=8, min_samples_leaf=1, max_samples=0.6, max_features=0.5, max_depth=30, criterion='squared_error', bootstrap=True, random_state=42)

compare_model_dp(data_set_brands, model_et)

,Train_MAE,Val_MAE,Gap_MAE_%,Train_R2,Val_R2
Data_Preprocessing,,,,,
Cheap,795.591804,986.058671,23.940275,0.959900,0.939594
Original,1086.620958,1336.502778,22.996227,0.961867,0.946629
Expensive,1581.916688,1923.424139,21.588207,0.945781,0.926281


In [ ]:
# Analysis for all brand subsets with GradientBoostingRegressor
model_gb = GradientBoostingRegressor(subsample=0.95, n_estimators=1000, min_samples_split=15, min_samples_leaf=5, max_features=0.7, max_depth=5, loss='huber', learning_rate=0.06, random_state=42)

compare_model_dp(data_set_brands, model_gb)

,Train_MAE,Val_MAE,Gap_MAE_%,Train_R2,Val_R2
Data_Preprocessing,,,,,
Cheap,834.905126,958.108699,14.756596,0.956005,0.945223
Original,1189.348326,1323.665549,11.293346,0.958551,0.950099
Expensive,1471.380186,1827.793378,24.223052,0.954282,0.932739


In [ ]:
# Analysis for all brand subsets with KNeighborsRegressor
model_knn = KNeighborsRegressor(n_neighbors=10, weights='uniform', p=1, metric='minkowski', leaf_size=20, algorithm='kd_tree')

compare_model_dp(data_set_brands, model_knn)

,Train_MAE,Val_MAE,Gap_MAE_%,Train_R2,Val_R2
Data_Preprocessing,,,,,
Cheap,932.103966,1052.682998,12.936221,0.943092,0.928615
Original,1301.857906,1447.494084,11.186795,0.942305,0.935130
Expensive,1920.741897,2120.028553,10.375504,0.914227,0.905612


**Effects of Brands Analysis Conclusion:** 

Across all four evaluated models, predictive performance is consistently higher for cheaper vehicles than for expensive ones. Cheap brand subsets achieve lower validation MAE and higher R² values across all models, while expensive subsets exhibit larger prediction errors and reduced generalization performance.

This performance gap can be explained by several factors. First, cheap brands contain a larger number of observations than expensive brands, which allows models with more data to learn stable and representative patterns. In contrast, expensive brands are represented by a smaller sample size, increasing model variance and reducing generalization capability.

Finally, residual data quality issues, particularly inconsistencies and misspellings in cars models name, are likely to introduce additional noise in categorical encodings. This effect disproportionately impacts expensive brands, which often include a wider range of specialized models and variants.

Overall, these results suggest that model performance differences across brand price segments are driven not only by model choice, but also by data distribution, sample size imbalance, and feature quality. While a single global model provides stable performance, predicting prices for expensive vehicles remains a more challenging task that may benefit from improved data cleaning or segment-specific modeling strategies.